In [ ]:
Website Crawler
      │
      ▼
HTML Cleaning
      │
      ▼
Text Chunking
      │
      ▼
Embeddings
      │
      ▼
Vector Database (Chroma)
      │
      ▼
Retriever
      │
      ▼
Groq LLM
      │
      ▼
Chatbot

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

BASE_URL = "https://www.genrocket.com"
START_URL = "https://www.genrocket.com/download-literature/"

visited = set()
pages = []

def is_internal(link):
    return urlparse(link).netloc == urlparse(BASE_URL).netloc

def crawl(url):
    if url in visited:
        return

    visited.add(url)

    try:
        res = requests.get(url, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        text = soup.get_text()
        pages.append({"url": url, "text": text})

        for a in soup.find_all("a", href=True):
            link = urljoin(url, a["href"])

            if is_internal(link):
                crawl(link)

    except Exception as e:
        print("Error:", e)

crawl(START_URL)

print("Total pages:", len(pages))

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

texts = [p["text"] for p in pages]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

documents = splitter.create_documents(texts)

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain.vectorstores import Chroma

vectordb = Chroma.from_documents(
    documents,
    embedding=embeddings,
    persist_directory="genrocket_db"
)

vectordb.persist()

In [ ]:
from langchain_groq import ChatGroq
import os

os.environ["GROQ_API_KEY"] = "<GROQ-KEY>"

llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0
)

In [ ]:
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k":4}
)

In [ ]:
from langchain.prompts import PromptTemplate

template = """
You are a QA assistant.

Answer ONLY from the provided context.

If the answer is not in the context say:
"I could not find the answer in the provided website data."

Context:
{context}

Question:
{question}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [ ]:
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)